# Demo Pipeline — Legal Contract Analyzer

Acest notebook demonstrează end-to-end sistemul de analiză juridică pe un contract real.

**Contract ales:** `data/draft_contract.pdf` — contract de prestări servicii cu clauze variate

**De ce l-am ales:** Conține clauze de penalități neplafonat, reziliere unilaterală și prelucrare date personale fără temei legal — tipuri de risc pe care sistemul ar trebui să le detecteze.

**Așteptări:** Cel puțin 2 clauze RIDICAT (penalitate + reziliere), 1-2 clauze MEDIU (date personale), restul SCAZUT sau CONFORM.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

print('Setup OK')

## 1. Rulăm graful LangGraph end-to-end

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(name)s | %(message)s')

from src.graph.workflow import run_workflow

PDF_PATH = str(ROOT / 'data' / 'draft_contract.pdf')

print(f'Rulez pipeline pe: {PDF_PATH}')
final_state = run_workflow(PDF_PATH)
print('\nPipeline finalizat!')

## 2. Starea finală a grafului

In [ ]:
parsed_doc = final_state['parsed_doc']
risk_map = final_state.get('risk_map', {})
recommendations = final_state.get('recommendations', [])

print(f'Document: {parsed_doc.metadata.title}')
print(f'Pagini: {parsed_doc.metadata.page_count}')
print(f'Secțiuni: {len(parsed_doc.sections)}')
print(f'Clauze: {len(parsed_doc.clauses)}')
print(f'High risk alert: {final_state.get("high_risk_alert", False)}')
print(f'Iterații retrieval: {final_state.get("iteration", 0)}')

from collections import Counter
risk_counts = Counter(dto.risk_level.value for dto in risk_map.values())
print(f'\nDistribuție risc: {dict(risk_counts)}')

## 3. Scoruri RAGAS — interpretare

In [ ]:
import json

rag_eval_path = ROOT / 'logs' / 'rag_evaluation.json'
if rag_eval_path.exists():
    with open(rag_eval_path) as f:
        rag_eval = json.load(f)
    
    print('Scoruri RAGAS:')
    scores = rag_eval.get('scores', {})
    for metric, score in scores.items():
        status = '✓' if score >= 0.5 else '✗'
        print(f'  {status} {metric}: {score:.3f}')
    
    print(f'\nInterpretare:')
    print(f'  faithfulness={scores.get("faithfulness", 0):.3f}: '
          f'LLM-ul ancorează bine în context (evită halucinațiile)' if scores.get('faithfulness', 0) >= 0.6
          else f'  faithfulness scăzut — LLM-ul generează referințe neancorate în corpus')
    print(f'  context_precision={scores.get("context_precision", 0):.3f}: '
          f'chunk-urile recuperate sunt relevante' if scores.get('context_precision', 0) >= 0.5
          else f'  context_precision scăzut — multe chunk-uri recuperate sunt irelevante')
else:
    print('logs/rag_evaluation.json nu există. Rulați scripts/evaluate_rag.py mai întâi.')

## 4. Vizualizări

In [ ]:
from IPython.display import Image, display

heatmap_path = ROOT / 'logs' / 'retrieval_heatmap.png'
if heatmap_path.exists():
    print('Heatmap retrieval (5 clauze × 5 chunk-uri):')
    display(Image(str(heatmap_path)))
else:
    print('retrieval_heatmap.png nu există. Rulați scripts/test_retrieval.py.')

In [ ]:
risk_dist_path = ROOT / 'logs' / 'risk_distribution.png'
if risk_dist_path.exists():
    print('Distribuția nivelurilor de risc:')
    display(Image(str(risk_dist_path)))
else:
    print('risk_distribution.png nu există. Rulați pipeline-ul mai întâi.')

In [ ]:
graph_path = ROOT / 'logs' / 'workflow_graph.png'
if graph_path.exists():
    print('Graful LangGraph:')
    display(Image(str(graph_path)))
else:
    print('workflow_graph.png nu există. Se generează la run_workflow().')

## 5. Raportul final

In [ ]:
from IPython.display import Markdown, display

report_path = ROOT / 'data' / 'raport_final.md'
if report_path.exists():
    report_text = report_path.read_text(encoding='utf-8')
    display(Markdown(report_text))
else:
    print('Raportul nu a fost generat încă.')

## 6. Concluzie

**Timpi per nod** (din logs/run_*.json):

In [ ]:
import glob

log_files = sorted(glob.glob(str(ROOT / 'logs' / 'run_*.json')), reverse=True)
if log_files:
    with open(log_files[0]) as f:
        run_log = json.load(f)
    
    total_sec = 0
    print('Timpi per nod (ultima rulare):')
    for entry in run_log:
        dur = entry.get('duration_sec', 0)
        total_sec += dur
        print(f"  {entry['node']}: {dur:.2f}s")
    print(f'\n  TOTAL: {total_sec:.2f}s')
else:
    print('Nu există fișiere de log. Rulați pipeline-ul mai întâi.')

print()
print('Cost estimat per agent (GPT-4o-mini):')
print('  Risk Assessment: ~$0.001/clauza (input ~2k tokens, output ~200 tokens)')
print('  Recommendation (MEDIU): ~$0.002/clauza')
print('  Recommendation (RIDICAT, self-consistency): ~$0.006/clauza (3+1 apeluri)')
print('  Embedding retrieval: ~$0.0001/clauza (text-embedding-3-small)')

print()
print('Limitări identificate:')
print('  1. Corpusul acoperă achizițiile publice (ANAP, Legea 98) dar nu contractele civile')
print('     comerciale → clauzele din contracte B2B private primesc des NECUNOSCUT.')
print('  2. Parser-ul bazat pe regex nu extrage corect clauzele din PDF-uri cu layout complex')
print('     (tabele, coloane multiple) → soluție: parser multimodal sau PyMuPDF.')
print('  3. Self-consistency adaugă 3x costul pentru clauze RIDICAT — la contracte mari')
print('     cu multe clauze riscante, costul poate fi prohibitiv.')

print()
print('Propunere de îmbunătățire:')
print('  Adăugarea unui Reranker (Cohere Rerank) după retrieval pentru a elimina')
print('  chunk-urile cu scor semantic bun dar irelevante juridic (ex: GDPR art. 17')
print('  recuperat pentru o clauză de penalitate din cauza vocabularului comun).')